In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
base_dir = '...'

In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
# Show versioning of deep learning libraries
import torch, torchvision
print(torch.__version__, torch.cuda.is_available())

2.9.0+cu126 True


In [ ]:
import os
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision as transforms
from torch.utils.data import Dataset, DataLoader, random_split, Subset
import json

init_filters = 32   # Initial number of filters
depth = 3           # Depth of the U-Net
input_size = (1024, 1024) # Target spatial resolution
in_channels = 3 # Changed from 1 to 3
out_channels = 1    # Output channels
n_epochs = 20 # Total number of training iterations over the dataset
batch_size = 4 # Number of samples processed before updating model weights
learning_rate = 1e-3 # Step size for the optimizer
checkpoint_freq = 1 # Frequency of saving model weights (every epoch)

# Paths for Training Data (Images and corresponding Masks)
train_img_dir = os.path.join(base_dir, 'train', 'image')
train_mask_dir = os.path.join(base_dir, 'train', 'manual_py')

# Paths for Validation Data
val_img_dir = os.path.join(base_dir, 'val', 'image')
val_mask_dir = os.path.join(base_dir, 'val', 'manual_py')

# Directory where training checkpoints (model weights) will be saved
checkpoint_dir = os.path.join("...", 'unet_training_checkpoints')

# Check if the checkpoint directory exists; if not, create it
if not os.path.exists(checkpoint_dir):
    os.makedirs(checkpoint_dir)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.BCEWithLogitsLoss()

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    """Applies two consecutive conv-batchnorm-relu layers"""
    def __init__(self, in_channels, out_channels):
        super(DoubleConv, self).__init__()
        self.double_conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.double_conv(x)

class UNet(nn.Module):
    def __init__(self,
                 in_channels=3,
                 out_channels=1,
                 init_filters=32,
                 depth=3,
                 bilinear=True):
        super(UNet, self).__init__()
        self.depth = depth
        self.down_layers = nn.ModuleList()
        self.up_layers = nn.ModuleList()
        self.pool = nn.MaxPool2d(2)
        # Encoder
        filters = init_filters
        for d in range(depth):
            conv = DoubleConv(in_channels, filters)
            self.down_layers.append(conv)
            in_channels = filters
            filters *= 2

        # Bottleneck
        self.bottleneck = DoubleConv(in_channels, filters)

        # Decoder
        for d in range(depth):
            filters //= 2
            if bilinear:
                up = nn.Sequential(
                    nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                    nn.Conv2d(filters * 2, filters, kernel_size=1)
                )
            else:
                up = nn.ConvTranspose2d(filters * 2, filters, kernel_size=2, stride=2)
            self.up_layers.append(nn.ModuleDict({
                'up': up,
                'conv': DoubleConv(filters * 2, filters)
            }))

        # Output layer
        self.out_conv = nn.Conv2d(init_filters, out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []
        for down in self.down_layers:
            x = down(x)
            skip_connections.append(x)
            x = self.pool(x)

        x = self.bottleneck(x)

        for i in range(self.depth):
            skip = skip_connections[-(i+1)]
            up = self.up_layers[i]['up'](x)
            if up.size() != skip.size():
                # Resize in case of odd size mismatch
                up = F.interpolate(up, size=skip.shape[2:])
            x = torch.cat([skip, up], dim=1)
            x = self.up_layers[i]['conv'](x)

        return self.out_conv(x)

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import torch.optim as optim

class DermaDataset(Dataset):
    def __init__(self, image_dir, mask_dir, transform=None):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.transform = transform
        self.image_list = sorted(os.listdir(image_dir))
        self.mask_list = sorted(os.listdir(mask_dir))
        assert len(self.image_list) == len(self.mask_list), "Number of images and masks do not match"

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_list[idx])
        mask_path = os.path.join(self.mask_dir, self.mask_list[idx])

        image = Image.open(img_path).convert('RGB')
        mask = Image.open(mask_path).convert('L')  # grayscale mask

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        # Convert mask to binary float tensor (0 or 1)
        mask = (mask > 0).float()

        return image, mask


# Transformations applied to both images and masks
transform = transforms.Compose([
    transforms.Resize(input_size),
    transforms.ToTensor(),
])

# Initialize datasets and corresponding dataloaders
train_dataset = DermaDataset(train_img_dir, train_mask_dir, transform=transform)
val_dataset = DermaDataset(val_img_dir, val_mask_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

In [ ]:
from sklearn.metrics import jaccard_score

# Inizializzazione Modello e Ottimizzatore
model = UNet(in_channels, out_channels, init_filters, depth).to(device)
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Checkpoints to resume incomplete training
start_epoch = 0

checkpoint_files = sorted([f for f in os.listdir(checkpoint_dir) if f.endswith('.pt')])
if checkpoint_files:
    last_checkpoint = os.path.join(checkpoint_dir, checkpoint_files[-1])
    print(f"Loading checkpoint: {last_checkpoint}")

    model.load_state_dict(torch.load(last_checkpoint, map_location=device))
    start_epoch = int(last_checkpoint.split('_')[-1].split('.')[0])
    print(f"Resuming training from epoch {start_epoch + 1}")
else:
    print("No checkpoint found, starting from scratch.")

# save
params = {
    "input_size": input_size, "in_channels": in_channels, "out_channels": out_channels,
    "init_filters": init_filters, "depth": depth, "n_epochs": n_epochs,
    "batch_size": batch_size, "learning_rate": learning_rate, "checkpoint_freq": checkpoint_freq,
}
params_path = os.path.join(checkpoint_dir, 'training_params.json')
with open(params_path, 'w') as f:
    json.dump(params, f, indent=4)

history = {'train_loss': [], 'val_loss': [], 'val_iou': []}

for epoch in range(start_epoch, n_epochs):

    # TRAINING FASE
    model.train()
    train_loss = 0.0

    for images, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{n_epochs} - Training"):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()
        outputs = model(images) # Logits

        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

    train_loss /= len(train_dataset)
    history['train_loss'].append(train_loss)
    print(f"Epoch {epoch+1} Train Loss: {train_loss:.4f}")

    # VALIDATION FASE
    model.eval()
    val_loss = 0.0
    all_preds = []
    all_masks = []

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)

            loss = criterion(outputs, masks)
            val_loss += loss.item() * images.size(0)

            if out_channels == 1:
                # Binary segmentation: sigmoid + threshold
                probs = torch.sigmoid(outputs)
                preds = (probs > 0.5).long()
                true = masks.long()
            else:
                # Multi-class segmentation: argmax
                preds = torch.argmax(outputs, dim=1)
                true = masks.long()

            all_preds.append(preds.cpu().numpy().flatten())
            all_masks.append(true.cpu().numpy().flatten())

    val_loss /= len(val_loader.dataset)

    # mIoU (Jaccard) metric
    y_true = np.concatenate(all_masks)
    y_pred = np.concatenate(all_preds)

    iou = jaccard_score(y_true, y_pred, average='macro')

    print(f"Epoch {epoch+1} Val Loss: {val_loss:.4f} - Val IoU: {iou:.4f}")

    # Save checkpoint every [checkpoint_freq] epochs
    if (epoch + 1) % checkpoint_freq == 0:
        checkpoint_path = os.path.join(checkpoint_dir, f"model_epoch_{epoch+1}.pt")
        torch.save(model.state_dict(), checkpoint_path)
        print(f"Checkpoint saved at {checkpoint_path}")

print("Training complete!")
